In [0]:
%run "../00_config"

In [0]:
%pip install pycountry

In [0]:
import pycountry
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

##Load Bronze Holidays

In [0]:
df_bronze = spark.table(BRONZE_HOLIDAYS)
display(df_bronze)

##Clean and transform

In [0]:
df_silver = (df_bronze
    # Cast types
    .withColumn("year",                   F.col("year").cast(IntegerType()))
    .withColumn("holiday_date",
        F.to_date(F.substring(F.col("holiday_date"), 1, 10), "yyyy-MM-dd")
    )

    # Replace empty strings with null
    .withColumn("holiday_name",
        F.when(F.col("holiday_name") == "", F.lit(None))
        .otherwise(F.col("holiday_name"))
    )
    .withColumn("holiday_type",
        F.when(F.col("holiday_type") == "", F.lit(None))
        .otherwise(F.col("holiday_type"))
    )
    .withColumn("holiday_description",
        F.when(F.col("holiday_description") == "", F.lit(None))
        .otherwise(F.col("holiday_description"))
    )

    # Clean date
    .withColumn("_snapshot_date", (F.to_date(F.col("_snapshot_date"), "dd-MM-yyyy")))

    # Add ingestion metadata
    .withColumn("_silver_ingested_at", F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat())))

    # Final column selection
    .select(
        "holiday_name",
        "holiday_type",
        "holiday_description",
        "holiday_date",
        "year",
        "country",
        "_snapshot_date",
        "_silver_ingested_at"
    )
)

print(f"Silver rows after cleaning: {df_silver.count()}")

## Dynamically convert ISO country codes to full names

In [0]:
country_codes = [
    row["country"]
    for row in df_silver.select("country").distinct().collect()
    if row["country"]
]

country_expr = F.col("country")
for code in country_codes:
    try:
        full_name = pycountry.countries.get(alpha_2=code.upper()).name
    except:
        full_name = code  
    country_expr = F.when(
        F.upper(F.col("country")) == code.upper(), full_name
    ).otherwise(country_expr)

df_silver = df_silver.withColumn("country", country_expr)
display(df_silver)

##Write to Silver


In [0]:
(df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_HOLIDAYS)
)

##Validate

In [0]:
df_check = spark.table(SILVER_HOLIDAYS)

print(f"Total rows  : {df_check.count()}")
print(f"Schema      :")
df_check.printSchema()

df_check.select(
    "holiday_name", "holiday_date", "holiday_type", "year"
).orderBy("holiday_date").show(15, truncate=40)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.silver.silver_holidays;